In [2]:
import pandas as pd
df = pd.read_csv("data/raw/Fire Incidents Data.csv", low_memory=False)
df.shape

(36564, 43)

In [3]:
list(df.columns)

['_id',
 'Incident_Number',
 'Initial_CAD_Event_Type',
 'Final_Incident_Type',
 'Exposures',
 'Incident_Station_Area',
 'Incident_Ward',
 'Intersection',
 'Latitude',
 'Longitude',
 'Property_Use',
 'Building_Status',
 'TFS_Alarm_Time',
 'TFS_Arrival_Time',
 'Ext_agent_app_or_defer_time',
 'Fire_Under_Control_Time',
 'Last_TFS_Unit_Clear_Time',
 'Number_of_responding_apparatus',
 'Number_of_responding_personnel',
 'Count_of_Persons_Rescued',
 'TFS_Firefighter_Casualties',
 'Civilian_Casualties',
 'Estimated_Number_Of_Persons_Displaced',
 'Estimated_Dollar_Loss',
 'Business_Impact',
 'Status_of_Fire_On_Arrival',
 'Method_Of_Fire_Control',
 'Area_of_Origin',
 'Level_Of_Origin',
 'Extent_Of_Fire',
 'Smoke_Spread',
 'Ignition_Source',
 'Material_First_Ignited',
 'Possible_Cause',
 'Smoke_Alarm_at_Fire_Origin',
 'Smoke_Alarm_at_Fire_Origin_Alarm_Failure',
 'Smoke_Alarm_at_Fire_Origin_Alarm_Type',
 'Smoke_Alarm_Impact_on_Persons_Evacuating_Impact_on_Evacuation',
 'Fire_Alarm_System_Presence'

In [4]:
cols = ["TFS_Alarm_Time", "TFS_Arrival_Time", "Latitude", "Longitude",
        "Incident_Station_Area", "Final_Incident_Type"]
df[cols].head(10)

,TFS_Alarm_Time,TFS_Arrival_Time,Latitude,Longitude,Incident_Station_Area,Final_Incident_Type
0,2018-02-24T21:04:29,2018-02-24T21:10:11,43.686558,-79.599419,441,01 - Fire
1,2018-02-24T21:24:43,2018-02-24T21:29:31,43.766135,-79.390039,116,01 - Fire
2,2018-02-25T13:29:59,2018-02-25T13:36:49,43.743230,-79.245061,221,"03 - NO LOSS OUTDOOR fire (exc: Sus.arson,vand..."
3,2018-02-25T14:13:39,2018-02-25T14:18:07,43.708659,-79.478062,133,01 - Fire
4,2018-02-25T18:20:43,2018-02-25T18:26:19,43.718118,-79.443184,132,"03 - NO LOSS OUTDOOR fire (exc: Sus.arson,vand..."
5,2018-02-25T18:31:19,2018-02-25T18:35:17,43.773787,-79.162282,215,01 - Fire
6,2018-02-26T18:18:55,2018-02-26T18:24:47,43.714811,-79.304107,235,01 - Fire
7,2018-02-27T10:28:12,2018-02-27T10:35:13,43.748582,-79.222367,231,01 - Fire
8,2018-02-25T15:48:34,2018-02-25T15:52:04,43.652152,-79.382336,332,01 - Fire
9,2018-02-26T15:32:11,2018-02-26T15:37:40,43.653882,-79.451846,426,"03 - NO LOSS OUTDOOR fire (exc: Sus.arson,vand..."


In [5]:
df[cols].isna().sum()

TFS_Alarm_Time           0
TFS_Arrival_Time         1
Latitude                 2
Longitude                2
Incident_Station_Area    1
Final_Incident_Type      0
dtype: int64

In [6]:
df["Final_Incident_Type"].value_counts()

Final_Incident_Type
01 - Fire                                                                                  26029
03 - NO LOSS OUTDOOR fire (exc: Sus.arson,vandal,child playing,recycling or dump fires)    10384
02 - Explosion (including during Fire, excluding Codes 3 & 11-13)                            151
Name: count, dtype: int64

In [7]:
df["Incident_Station_Area"].head(5).tolist()

['441', '116', '221', '133', '132']

In [8]:
df["alarm"] = pd.to_datetime(df["TFS_Alarm_Time"])
df["arrival"] = pd.to_datetime(df["TFS_Arrival_Time"])
df["wait_s"] = (df["arrival"] - df["alarm"]).dt.total_seconds()
df["wait_s"].describe()

count    36563.000000
mean       324.857971
std        262.548113
min      -2245.000000
25%        254.000000
50%        309.000000
75%        375.000000
max      40942.000000
Name: wait_s, dtype: float64

In [9]:
print("negative waits:", (df["wait_s"] < 0).sum())
print("waits over 1 hour:", (df["wait_s"] > 3600).sum())

negative waits: 2
waits over 1 hour: 5


In [10]:
raw = df["Incident_Station_Area"]
print("distinct raw values:", raw.nunique())
print("distinct after trimming:", raw.str.strip().nunique())
print("rows with extra spaces:", (raw.str.len() != raw.str.strip().str.len()).sum())

distinct raw values: 170
distinct after trimming: 87
rows with extra spaces: 10705


In [11]:
df.groupby(df["alarm"].dt.year)["Final_Incident_Type"].apply(lambda x: x.str.startswith("03").sum())

alarm
2011       0
2012       0
2013       0
2014       0
2015       0
2016       0
2017       0
2018    1348
2019    1078
2020    1552
2021    1607
2022    1627
2023    1580
2024    1592
Name: Final_Incident_Type, dtype: int64

# Stage 1 summary: what I learned from exploring the incidents file

**File:** Fire Incidents Data.csv, 36,564 rows and 43 columns. Each row is one fire or explosion incident, with alarm times from 2011 to 2024.

## Columns I will use
TFS_Alarm_Time, TFS_Arrival_Time, Latitude, Longitude, Incident_Station_Area, Final_Incident_Type. The other 37 columns (ignition source, smoke alarms, and so on) are not needed to measure response time.

## Outcome I will measure
Response time = TFS_Arrival_Time minus TFS_Alarm_Time, in seconds. It includes dispatch and turnout time as well as driving, so it is not pure travel time.

## What I found
- **Missing values are very few:** 1 arrival time, 2 latitudes, 2 longitudes and 1 station area are empty.
- **Incident types:** 26,029 are 01 - Fire, 151 are 02 - Explosion and 10,384 are 03 - No loss outdoor fire. Type 03 does not exist before 2018, so including it would change the data partway through the period.
- **Typical wait:** median 309 s (about 5 min 9 s). The middle half of incidents waited between 254 s and 375 s.
- **Impossible values:** 2 negative waits (the smallest is -2,245 s) and 5 waits over one hour (the largest is 40,942 s). These are almost certainly recording errors. The mean (325 s) is pulled up by them, so I will report medians and percentiles.
- **Station codes are inconsistent:** 10,705 rows have codes padded with extra spaces. There are 170 distinct raw values, but only 87 after trimming. The remaining values include an empty value and an odd code, 145P.

## Decisions for Stage 2 (cleaning)
1. Keep only incident types 01 and 02, for consistency across the years.
2. Remove rows with a missing arrival time, a negative wait, an implausibly short wait, or a wait over one hour, and keep a log of how many rows each step removes.
3. Trim spaces from the station codes.
4. Keep the incidents without coordinates, but flag them, so I can test whether missing locations bias the results.

## Some Important Points:
1. **Here, I've used the median** because a few recording errors, such as a wait of over 11 hours or a negative one, pull the average up or down a lot. The median is the middle value, so a handful of extreme rows barely moves it. It describes the typical incident better. I will also report percentiles, such as the 90th, to show the slow end.
2. **I've leave out type 03** because it only appears from 2018 onward (zero rows before that). Including it would mean the data changes character partway through the 14 years, so any trend over time would partly reflect a change in what is recorded, not a change in response. Types 01 and 02 exist for the whole period.